In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from PIL import Image
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModel
from tqdm import tqdm
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pickle

In [18]:
print("Loading Flickr8k dataset...")
dataset = load_dataset('jxie/flickr8k')
print(f"Dataset structure: {dataset}")
print(f"Dataset keys: {dataset.keys()}")

Loading Flickr8k dataset...
Dataset structure: DatasetDict({
    train: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 6000
    })
    validation: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
})
Dataset keys: dict_keys(['train', 'validation', 'test'])


In [19]:
print(f"Train set columns: {dataset['train'].column_names}")
print(f"Validation set columns: {dataset['validation'].column_names}")
print(f"Test set columns: {dataset['test'].column_names}")
print(f"\nTrain set size: {len(dataset['train'])}")
print(f"Validation set size: {len(dataset['validation'])}")
print(f"Test set size: {len(dataset['test'])}")
print(f"\nFirst training sample:")
first_sample = dataset['train'][0]
for key in first_sample.keys():
    print(f"{key}: ", end="")
    value = first_sample[key]
    if isinstance(value, Image.Image):
        print(f"Image of size {value.size}")
    else:
        print(f"{value}")

Train set columns: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4']
Validation set columns: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4']
Test set columns: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4']

Train set size: 6000
Validation set size: 1000
Test set size: 1000

First training sample:
image: Image of size (500, 399)
caption_0: A black dog is running after a white dog in the snow .
caption_1: Black dog chasing brown dog through snow
caption_2: Two dogs chase each other across the snowy ground .
caption_3: Two dogs play together in the snow .
caption_4: Two dogs running through a low lying body of water .


In [21]:
def preprocess_caption(caption):
    caption = caption.lower()
    caption = re.sub(f'[{re.escape(string.punctuation)}]', '', caption)
    caption = ' '.join(caption.split())
    return caption

# Test the preprocessing function on a sample caption
test_caption = "A dog is    running in the park!"
# Call preprocessing function on test caption
cleaned = preprocess_caption(test_caption)
# Print the original caption
print(f"Original: {test_caption}")
# Print the cleaned caption
print(f"Cleaned: {cleaned}")

Original: A dog is    running in the park!
Cleaned: a dog is running in the park


In [22]:
# Define special tokens for the vocabulary
SOS_TOKEN = '<SOS>'  # SOS token marks the start of a sequence
EOS_TOKEN = '<EOS>'  # EOS token marks the end of a sequence
PAD_TOKEN = '<PAD>'  # PAD token is used to pad sequences to the same length
UNK_TOKEN = '<UNK>'  # UNK token represents unknown words not in vocabulary
MAX_VOCAB_SIZE = 10000  # Set maximum vocabulary size to limit memory usage

# Define function to build vocabulary from captions
def build_vocabulary(dataset, max_size=10000):
    word_freq = Counter()  # Initialize counter to count word frequencies
    for split in ['train', 'validation', 'test']:  # Iterate through all captions in train, validation, and test sets
        for sample in tqdm(dataset[split], desc=f"Building vocab from {split}"):
            for i in range(5):   # Iterate through all caption fields (caption_0 to caption_4)
                caption_key = f'caption_{i}'
                caption = sample[caption_key]
                caption = preprocess_caption(caption)
                words = caption.split()
                word_freq.update(words)
    most_common_words = word_freq.most_common(max_size) # Get the most common words up to max_size
    most_common_words = [word for word, freq in most_common_words] # Extract just the words (ignore frequencies)


    word2idx = {  # Create word to index mapping starting from special tokens
        PAD_TOKEN: 0,  # Index 0 is reserved for PAD token
        UNK_TOKEN: 1,  # Index 1 is reserved for UNK token
        SOS_TOKEN: 2,  # Index 2 is reserved for SOS token
        EOS_TOKEN: 3,} # Index 3 is reserved for EOS token
    # Add common words to vocabulary starting from index 4
    for idx, word in enumerate(most_common_words, start=4):
        # Map each word to a unique index
        word2idx[word] = idx
    # Create reverse mapping from index to word
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word # Return both mappings

In [23]:
print("Building vocabulary...")
word2idx, idx2word = build_vocabulary(dataset, max_size=MAX_VOCAB_SIZE)
print(f"Vocabulary size: {len(word2idx)}")
print(f"First 20 words: {list(word2idx.items())[:20]}")

Building vocabulary...


Building vocab from test: 100%|██████████| 1000/1000 [00:02<00:00, 374.51it/s]

Vocabulary size: 8724
First 20 words: [('<PAD>', 0), ('<UNK>', 1), ('<SOS>', 2), ('<EOS>', 3), ('a', 4), ('in', 5), ('the', 6), ('on', 7), ('is', 8), ('and', 9), ('dog', 10), ('with', 11), ('man', 12), ('of', 13), ('two', 14), ('white', 15), ('black', 16), ('boy', 17), ('are', 18), ('woman', 19)]


In [24]:
def encode_caption(caption, word2idx, max_length=50):
    tokens = [word2idx[SOS_TOKEN]]  # Start with SOS token index
    caption = preprocess_caption(caption) # Preprocess the caption
    words = caption.split()
    for word in words:
        if word in word2idx: # If word is in vocabulary, add its index
            tokens.append(word2idx[word]) # Add word index to tokens
        else:
            tokens.append(word2idx[UNK_TOKEN])  # If word is not in vocabulary, add UNK token index
    tokens.append(word2idx[EOS_TOKEN])   # Add EOS token index at the end
    if len(tokens) > max_length:  # Truncate or pad to max_length
        tokens = tokens[:max_length]   # Truncate or pad to max_length
    else:
        tokens = tokens + [word2idx[PAD_TOKEN]] * (max_length - len(tokens))
    return np.array(tokens, dtype=np.int64)   # Convert to numpy array and return

In [25]:
# Define custom Dataset class for loading images and captions
class Flickr8kDataset(Dataset):

    def __init__(self, raw_dataset, word2idx, split='train', max_caption_length=50):
        self.dataset = raw_dataset[split] # Store the raw dataset
        self.word2idx = word2idx   # Store word to index mapping
        self.split = split        # Store split name (train/validation/test)
        self.max_caption_length = max_caption_length  # Store maximum caption length for encoding

    # Return the number of samples in the dataset
    def __len__(self):
        return len(self.dataset)

    # Get a single sample by index
    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image = sample['image']
        if image.mode != 'RGB':
            image = image.convert('RGB')
        caption = sample['caption_0']  # Store maximum caption length for encoding
        caption_encoded = encode_caption(caption, self.word2idx, self.max_caption_length) # Encode caption as indices
        return {'image': image, 'caption': caption_encoded, 'caption_text': caption}       # Return image and encoded caption

In [26]:
# Create dataset instances for each split
print("Creating dataset instances...")
train_dataset = Flickr8kDataset(dataset, word2idx, split='train')
val_dataset = Flickr8kDataset(dataset, word2idx, split='validation')
test_dataset = Flickr8kDataset(dataset, word2idx, split='test')
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Get a sample to verify everything works
print("\nSample from training dataset:")
sample = train_dataset[0]
print(f"Image size: {sample['image'].size}")
print(f"Caption shape: {sample['caption'].shape}")
print(f"Caption text: {sample['caption_text']}")
print(f"First 10 caption tokens: {sample['caption'][:10]}")

Creating dataset instances...
Train dataset size: 6000
Validation dataset size: 1000
Test dataset size: 1000

Sample from training dataset:
Image size: (500, 399)
Caption shape: (50,)
Caption text: A black dog is running after a white dog in the snow .
First 10 caption tokens: [  2   4  16  10   8  33 269   4  15  10]


In [27]:
# Set device to GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
# Load Vision Transformer image processor for preprocessing
print("Loading Vision Transformer...")
image_processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
vit_model = AutoModel.from_pretrained('google/vit-base-patch16-224', output_hidden_states=True)
vit_model = vit_model.to(device)
vit_model.eval()
print(f"ViT Model: {vit_model.config.model_type}")
print(f"Hidden size (embedding dimension): {vit_model.config.hidden_size}")
for param in vit_model.parameters(): # Disable gradient computation for all parameters
    param.requires_grad = False

Using device: cuda
Loading Vision Transformer...


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViT Model: vit
Hidden size (embedding dimension): 768


In [28]:
def extract_patch_tokens(images, vit_model, image_processor, device):
    """
    Extract patch token embeddings from Vision Transformer.
    Returns: Tensor of shape [batch_size, num_patches, hidden_size]
    For ViT-base-patch16-224: num_patches = 196, hidden_size = 768
    """
    with torch.no_grad():     # Disable gradient computation for efficiency
        inputs = image_processor(images, return_tensors='pt') # Preprocess images using the ViT image processor
        inputs = {k: v.to(device) for k, v in inputs.items()}  # Move preprocessed inputs to device
        outputs = vit_model(**inputs, output_hidden_states=True) # Pass images through ViT model to get encoder outputs
        # Extract last hidden state (shape: [batch_size, seq_length, hidden_size])
        # seq_length includes CLS token + 196 patch tokens = 197
        last_hidden_state = outputs.last_hidden_state
        # Extract patch tokens only (exclude CLS token at index 0)
        # Shape: [batch_size, 196, 768]
        patch_tokens = last_hidden_state[:, 1:, :]
        return patch_tokens

In [29]:
print("Testing patch token extraction...")
# Get first 2 samples from training dataset for testing
sample_images = [train_dataset[0]['image'], train_dataset[1]['image']]
# Extract patch tokens from sample images
patch_tokens = extract_patch_tokens(sample_images, vit_model, image_processor, device)
print(f"Patch tokens shape: {patch_tokens.shape}")
print(f"Expected shape: [2, 196, 768]")

Testing patch token extraction...
Patch tokens shape: torch.Size([2, 196, 768])
Expected shape: [2, 196, 768]


In [30]:
# Define function to extract CLS token from ViT
def extract_cls_token(images, vit_model, image_processor, device):
    """
    Extract CLS token embedding from Vision Transformer.
    Returns: Tensor of shape [batch_size, 1, hidden_size]
    For ViT-base-patch16-224: hidden_size = 768
    """

    with torch.no_grad():   # Disable gradient computation for efficiency
        inputs = image_processor(images, return_tensors='pt')  # Preprocess images using the ViT image processor
        inputs = {k: v.to(device) for k, v in inputs.items()}   # Move preprocessed inputs to device
        outputs = vit_model(**inputs, output_hidden_states=True) # Pass images through ViT model to get encoder outputs
        # Extract last hidden state (shape: [batch_size, seq_length, hidden_size])
        last_hidden_state = outputs.last_hidden_state
        # Extract CLS token only (index 0)
        # Shape: [batch_size, 1, 768]
        cls_token = last_hidden_state[:, 0:1, :]
        return cls_token

In [31]:
print("Testing CLS token extraction...")
cls_tokens = extract_cls_token(sample_images, vit_model, image_processor, device)
print(f"CLS tokens shape: {cls_tokens.shape}")
print(f"Expected shape: [2, 1, 768]")

Testing CLS token extraction...
CLS tokens shape: torch.Size([2, 1, 768])
Expected shape: [2, 1, 768]


In [32]:
# Define projection layer to map ViT embeddings to decoder dimension
class ViTProjection(nn.Module):
    def __init__(self, input_dim=768, output_dim=512, strategy='patch'):
        super(ViTProjection, self).__init__()  # Call parent class constructor
        self.strategy = strategy  # Store strategy type (patch or cls)
        self.projection = nn.Linear(input_dim, output_dim) # Create linear layer to project from input_dim to output_dim

    # Forward pass through projection layer
    def forward(self, embeddings):
        """
        Project embeddings to decoder dimension.
        Input shape: [batch_size, seq_length, 768]
        Output shape: [batch_size, seq_length, output_dim]
        """
        # Apply linear projection to embeddings
        projected = self.projection(embeddings)
        # Return projected embeddings
        return projected

In [34]:
# Create projection layers for both strategies
print("Creating projection layers...")
DECODER_DIM = 512  # Set decoder dimension (target embedding size)
projection_patch = ViTProjection(input_dim=768, output_dim=DECODER_DIM, strategy='patch') # Create projection layer for patch token strategy
projection_cls = ViTProjection(input_dim=768, output_dim=DECODER_DIM, strategy='cls') # Create projection layer for CLS token strategy
# Move projection layers to device
projection_patch = projection_patch.to(device)
projection_cls = projection_cls.to(device)
# Print projection layer information
print(f"Projection layers created with input_dim=768, output_dim={DECODER_DIM}")

print("\nTesting projection layers...")
projected_patch = projection_patch(patch_tokens)
print(f"Projected patch tokens shape: {projected_patch.shape}")
projected_cls = projection_cls(cls_tokens)
print(f"Projected CLS tokens shape: {projected_cls.shape}")

Creating projection layers...
Projection layers created with input_dim=768, output_dim=512

Testing projection layers...
Projected patch tokens shape: torch.Size([2, 196, 512])
Projected CLS tokens shape: torch.Size([2, 1, 512])


In [37]:
#summary of both strategies
print("SUMMARY:")
print("Strategy A (Patch Tokens):")
print("  - Uses 196 patch token embeddings (one for each 16x16 patch)")
print("  - Preserves spatial information from the image")
print("  - Enables fine-grained cross-attention between image regions and caption words")
print("  - Allows visualization of attention heatmaps as 14x14 grids")
print("  - Higher memory and compute requirements")
print("\nStrategy B (CLS Token):")
print("  - Uses only the global CLS token embedding")
print("  - Loses spatial information from the image")
print("  - Simple baseline with lower memory requirements")
print("  - Cannot visualize spatial attention patterns")
print("  - Better for resource-constrained environments")

SUMMARY:
Strategy A (Patch Tokens):
  - Uses 196 patch token embeddings (one for each 16x16 patch)
  - Preserves spatial information from the image
  - Enables fine-grained cross-attention between image regions and caption words
  - Allows visualization of attention heatmaps as 14x14 grids
  - Higher memory and compute requirements

Strategy B (CLS Token):
  - Uses only the global CLS token embedding
  - Loses spatial information from the image
  - Simple baseline with lower memory requirements
  - Cannot visualize spatial attention patterns
  - Better for resource-constrained environments


# Part 3 – Transformer Decoder



## 3.1 Positional Encoding



In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    """
    Injects position information into token embeddings.
    Uses fixed sine/cosine waves — nothing is learned here.

    Args:
        d_model   : embedding dimension (must match decoder hidden size)
        max_len   : maximum caption length we will ever see
        dropout   : regularisation dropout applied after adding PE
    """
    def __init__(self, d_model: int, max_len: int = 200, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Build the PE table once and store as a buffer (not a parameter)
        pe = torch.zeros(max_len, d_model)          # [max_len, d_model]
        position = torch.arange(0, max_len).unsqueeze(1).float()  # [max_len, 1]

        # Frequency term: 1 / 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # even indices → sin
        pe[:, 1::2] = torch.cos(position * div_term)   # odd  indices → cos

        pe = pe.unsqueeze(0)   # [1, max_len, d_model]  — batch dimension
        self.register_buffer('pe', pe)  # saved in state_dict but not trained

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : [batch, seq_len, d_model]
        Returns x + positional encoding, same shape.
        """
        x = x + self.pe[:, :x.size(1), :]   # broadcast over batch
        return self.dropout(x)


# Quick sanity-check
pe_test = PositionalEncoding(d_model=512, max_len=200)
dummy    = torch.zeros(4, 30, 512)          # batch=4, seq=30, dim=512
out      = pe_test(dummy)
print(f'PE output shape: {out.shape}  (expected [4, 30, 512])')
print('Positional encoding ready ✓')


PE output shape: torch.Size([4, 30, 512])  (expected [4, 30, 512])
Positional encoding ready ✓


## 3.2 Single Transformer Decoder Layer



In [ ]:
class TransformerDecoderLayer(nn.Module):
    """
    One full decoder block:
      1. Masked multi-head self-attention
      2. Multi-head cross-attention  (queries=decoder, keys/values=encoder memory)
      3. Position-wise feed-forward network

    Args:
        d_model   : embedding / hidden dimension (512)
        num_heads : number of attention heads (8 → each head has dim 64)
        d_ff      : inner feed-forward dimension (2048)
        dropout   : dropout probability
    """
    def __init__(self, d_model: int = 512, num_heads: int = 8,
                 d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()

        # ── Sub-layer 1: Masked Self-Attention ──────────────────────────────
        # batch_first=True means tensors are [Batch, Seq, Dim] (more intuitive)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(d_model)

        # ── Sub-layer 2: Cross-Attention ────────────────────────────────────
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(d_model)

        # ── Sub-layer 3: Feed-Forward Network ───────────────────────────────
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    # ── helper: build causal (upper-triangular) mask ─────────────────────────
    @staticmethod
    def _causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
        """
        Returns an additive mask of shape [seq_len, seq_len].
        Upper-triangle is -inf (forbidden future), diagonal & below are 0 (allowed past).

        Example for seq_len=4:
          [[  0, -inf, -inf, -inf],
           [  0,    0, -inf, -inf],
           [  0,    0,    0, -inf],
           [  0,    0,    0,    0]]
        """
        mask = torch.triu(
            torch.full((seq_len, seq_len), float('-inf'), device=device),
            diagonal=1
        )
        return mask

    def forward(
        self,
        tgt: torch.Tensor,           # [B, T, d_model]  decoder input (caption so far)
        memory: torch.Tensor,        # [B, S, d_model]  encoder output (image patches)
        tgt_key_padding_mask=None,   # [B, T] True where PAD tokens are
    ) -> torch.Tensor:

        T = tgt.size(1)
        causal_mask = self._causal_mask(T, tgt.device)  # [T, T]

        # 1. Masked Self-Attention: each word attends only to past words
        attn_out, _ = self.self_attn(
            query=tgt, key=tgt, value=tgt,
            attn_mask=causal_mask,              # prevents peeking ahead
            key_padding_mask=tgt_key_padding_mask
        )
        tgt = self.norm1(tgt + self.dropout(attn_out))  # residual + norm

        # 2. Cross-Attention: decoder queries the image patch memory
        cross_out, _ = self.cross_attn(
            query=tgt,      # questions come from caption tokens
            key=memory,     # keys from image patches
            value=memory    # values from image patches
        )
        tgt = self.norm2(tgt + self.dropout(cross_out))  # residual + norm

        # 3. Feed-Forward Network
        ffn_out = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout(ffn_out))    # residual + norm

        return tgt   # [B, T, d_model]


# Quick test
layer  = TransformerDecoderLayer(d_model=512, num_heads=8, d_ff=2048)
tgt_t  = torch.zeros(2, 10, 512)   # batch=2, seq=10
mem_t  = torch.zeros(2, 196, 512)  # batch=2, 196 patches
out_t  = layer(tgt_t, mem_t)
print(f'Decoder layer output shape: {out_t.shape}  (expected [2, 10, 512])')

# Verify the causal mask
mask = TransformerDecoderLayer._causal_mask(5, torch.device('cpu'))
print('\nCausal mask (5×5):')
print(mask)


Decoder layer output shape: torch.Size([2, 10, 512])  (expected [2, 10, 512])

Causal mask (5×5):
tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])


## 3.3 Full Transformer Decoder (stacked layers)




In [ ]:
class TransformerDecoder(nn.Module):
    """
    Full Transformer Decoder for image captioning.

    Architecture:
      Embedding  →  Positional Encoding  →  N × DecoderLayer  →  Linear(vocab)

    Args:
        vocab_size  : total number of unique tokens (8724)
        d_model     : embedding / hidden dimension (512)
        num_heads   : attention heads per layer (8)
        num_layers  : number of stacked decoder layers (3)
        d_ff        : feed-forward inner dimension (2048)
        max_len     : maximum caption length (50)
        dropout     : dropout probability (0.1)
        pad_idx     : vocabulary index of <PAD> (0) — ignored in embedding
    """
    def __init__(
        self,
        vocab_size: int,
        d_model: int     = 512,
        num_heads: int   = 8,
        num_layers: int  = 3,
        d_ff: int        = 2048,
        max_len: int     = 50,
        dropout: float   = 0.1,
        pad_idx: int     = 0,
    ):
        super().__init__()
        self.d_model  = d_model
        self.pad_idx  = pad_idx

        # ── Token Embedding ─────────────────────────────────────────────────
        # Maps each word index → d_model-dimensional vector
        # padding_idx=pad_idx means <PAD> tokens always map to the zero vector
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_idx
        )

        # ── Positional Encoding ─────────────────────────────────────────────
        self.pos_enc = PositionalEncoding(d_model=d_model, max_len=max_len + 10, dropout=dropout)

        # ── Stack of N Decoder Layers ────────────────────────────────────────
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # ── Output Projection ────────────────────────────────────────────────
        # Maps d_model → vocab_size (raw logits, NOT softmax)
        # CrossEntropyLoss expects raw logits, so we never apply softmax here
        self.output_proj = nn.Linear(d_model, vocab_size)

        # ── Weight Tying (optional but common) ──────────────────────────────
        # Share weights between input embedding and output projection
        # This saves params and often improves performance
        self.output_proj.weight = self.embedding.weight

        self._init_weights()

    def _init_weights(self):
        """Xavier uniform initialisation for all linear layers."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=d_model ** -0.5)
                module.weight.data[self.pad_idx].zero_()   # keep PAD = 0 vector

    def forward(
        self,
        tgt_tokens: torch.Tensor,   # [B, T]  word indices (teacher-forced captions)
        memory: torch.Tensor,       # [B, S, d_model]  projected image features
    ) -> torch.Tensor:
        """
        Forward pass (used during training with teacher forcing).

        Returns:
            logits : [B, T, vocab_size] — raw (un-normalised) scores for each token position.
                     Apply CrossEntropyLoss against the target caption shifted left by 1.
        """
        # Build padding mask: True where token == <PAD> (these positions are ignored)
        pad_mask = (tgt_tokens == self.pad_idx)   # [B, T]  bool

        # Embed tokens and add positional encoding
        # Scale embedding by sqrt(d_model) as in original Transformer paper
        x = self.embedding(tgt_tokens) * math.sqrt(self.d_model)  # [B, T, d_model]
        x = self.pos_enc(x)                                         # [B, T, d_model]

        # Pass through each decoder layer
        for layer in self.layers:
            x = layer(x, memory, tgt_key_padding_mask=pad_mask)

        # Project to vocabulary
        logits = self.output_proj(x)   # [B, T, vocab_size]
        return logits


# ── Instantiate both decoders (one for each encoder strategy) ───────────────
VOCAB_SIZE = len(word2idx)   # 8724
d_model    = DECODER_DIM     # 512 (defined in Part 2)

decoder_patch = TransformerDecoder(
    vocab_size=VOCAB_SIZE, d_model=d_model,
    num_heads=8, num_layers=3, d_ff=2048,
    max_len=50, dropout=0.1, pad_idx=word2idx['<PAD>']
).to(device)

decoder_cls = TransformerDecoder(
    vocab_size=VOCAB_SIZE, d_model=d_model,
    num_heads=8, num_layers=3, d_ff=2048,
    max_len=50, dropout=0.1, pad_idx=word2idx['<PAD>']
).to(device)

# Count trainable parameters
n_patch = sum(p.numel() for p in decoder_patch.parameters() if p.requires_grad)
print(f'Patch decoder trainable parameters: {n_patch:,}')
print(f'CLS   decoder trainable parameters: {sum(p.numel() for p in decoder_cls.parameters() if p.requires_grad):,}')

# Quick shape test
tgt_dummy = torch.randint(0, VOCAB_SIZE, (2, 30)).to(device)   # batch=2, seq=30
mem_dummy = torch.randn(2, 196, 512).to(device)                 # patch memory
logits    = decoder_patch(tgt_dummy, mem_dummy)
print(f'\nOutput logits shape: {logits.shape}  (expected [2, 30, {VOCAB_SIZE}])')


Patch decoder trainable parameters: 17,087,508
CLS   decoder trainable parameters: 17,087,508

Output logits shape: torch.Size([2, 30, 8724])  (expected [2, 30, 8724])


## 3.4 Complete Captioning Model



In [ ]:
class CaptionModel(nn.Module):
    """
    End-to-end captioning model:
      - vit_projection : Part-2 projection (768 → 512)
      - decoder        : Transformer decoder (Part 3)

    The ViT itself stays frozen outside this module.
    """
    def __init__(self, projection: nn.Module, decoder: TransformerDecoder):
        super().__init__()
        self.projection = projection
        self.decoder    = decoder

    def forward(
        self,
        image_features: torch.Tensor,   # [B, S, 768]  raw ViT features
        tgt_tokens: torch.Tensor,        # [B, T]       teacher-forced caption tokens
    ) -> torch.Tensor:
        """
        Returns logits [B, T, vocab_size].
        """
        memory = self.projection(image_features)   # [B, S, 512]
        logits = self.decoder(tgt_tokens, memory)  # [B, T, vocab_size]
        return logits


# Create the two complete models
model_patch = CaptionModel(projection_patch, decoder_patch).to(device)
model_cls   = CaptionModel(projection_cls,   decoder_cls  ).to(device)

print('CaptionModel (patch strategy) created ✓')
print('CaptionModel (CLS   strategy) created ✓')

# End-to-end shape test
feat_patch = torch.randn(2, 196, 768).to(device)  # simulated patch features
feat_cls   = torch.randn(2,   1, 768).to(device)  # simulated CLS  feature
cap_in     = torch.randint(0, VOCAB_SIZE, (2, 30)).to(device)

out_patch = model_patch(feat_patch, cap_in)
out_cls   = model_cls  (feat_cls,   cap_in)
print(f'\nPatch model output: {out_patch.shape}  (expected [2, 30, {VOCAB_SIZE}])')
print(f'CLS   model output: {out_cls.shape}  (expected [2, 30, {VOCAB_SIZE}])')


CaptionModel (patch strategy) created ✓
CaptionModel (CLS   strategy) created ✓

Patch model output: torch.Size([2, 30, 8724])  (expected [2, 30, 8724])
CLS   model output: torch.Size([2, 30, 8724])  (expected [2, 30, 8724])


## 3.5 Inference Strategies

During **training**, we use *teacher forcing* — the ground-truth words are fed as input.  
During **inference**, no ground truth exists, so the model must generate words one by one.

### Strategy 1 — Greedy Decoding
At every step, pick the word with the **highest probability**. Fast and simple, but can get stuck in repetitive or suboptimal sequences.

```
<SOS> → "a" → "dog" → "is" → "running" → <EOS>
         ↑      ↑       ↑       ↑
       argmax argmax argmax argmax
```

### Strategy 2 — Beam Search
Instead of always picking the single best word, keep **B candidate sequences** (the beam) at each step and expand all of them. At the end, pick the sequence with the highest total log-probability.

- Wider beam → better quality, but slower (O(B × vocab) per step)
- `beam_size=1` reduces to greedy search


In [ ]:
def greedy_decode(
    model: CaptionModel,
    image_features: torch.Tensor,  # [1, S, 768] single image features
    word2idx: dict,
    idx2word: dict,
    max_len: int = 50,
) -> str:
    """
    Generate a caption for one image using greedy decoding.
    At each step we take argmax of the logits at the last position.

    Returns: the generated caption as a string.
    """
    model.eval()
    device = image_features.device

    sos_idx = word2idx['<SOS>']
    eos_idx = word2idx['<EOS>']

    # Start with just the <SOS> token
    generated = [sos_idx]   # list of token indices produced so far

    with torch.no_grad():
        # Project image features once (stays constant throughout generation)
        memory = model.projection(image_features)   # [1, S, 512]

        for _ in range(max_len):
            # Build current input tensor from generated tokens
            tgt = torch.tensor([generated], dtype=torch.long, device=device)  # [1, t]

            # Run decoder: logits shape = [1, t, vocab_size]
            logits = model.decoder(tgt, memory)

            # Take logits at the LAST position (the next-word prediction)
            next_logits = logits[0, -1, :]          # [vocab_size]
            next_token  = next_logits.argmax().item()  # pick best word

            generated.append(next_token)

            # Stop if <EOS> is produced
            if next_token == eos_idx:
                break

    # Convert indices back to words, skip special tokens
    words = [
        idx2word.get(idx, '<UNK>')
        for idx in generated
        if idx not in {word2idx['<SOS>'], word2idx['<EOS>'], word2idx['<PAD>']}
    ]
    return ' '.join(words)


def beam_search_decode(
    model: CaptionModel,
    image_features: torch.Tensor,  # [1, S, 768]
    word2idx: dict,
    idx2word: dict,
    beam_size: int = 5,
    max_len: int   = 50,
) -> str:
    """
    Generate a caption using beam search.

    The beam keeps the top-`beam_size` partial sequences ranked by their
    cumulative log-probability.  Completed sequences (ending with <EOS>)
    are stored separately and the best one is returned.

    Returns: the best generated caption as a string.
    """
    model.eval()
    device = image_features.device

    sos_idx = word2idx['<SOS>']
    eos_idx = word2idx['<EOS>']

    # Each beam entry: (cumulative_log_prob, list_of_token_indices)
    beam = [(0.0, [sos_idx])]
    completed = []

    with torch.no_grad():
        memory = model.projection(image_features)   # [1, S, 512]

        for _ in range(max_len):
            candidates = []   # all expanded candidates this step

            for score, tokens in beam:
                # Skip already-finished beams
                if tokens[-1] == eos_idx:
                    completed.append((score, tokens))
                    continue

                tgt = torch.tensor([tokens], dtype=torch.long, device=device)  # [1, t]
                logits = model.decoder(tgt, memory)                             # [1, t, V]
                log_probs = F.log_softmax(logits[0, -1, :], dim=-1)             # [V]

                # Expand: take top beam_size next tokens
                top_log_probs, top_ids = log_probs.topk(beam_size)
                for lp, idx in zip(top_log_probs.tolist(), top_ids.tolist()):
                    candidates.append((score + lp, tokens + [idx]))

            if not candidates:
                break

            # Keep only the best beam_size sequences
            beam = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_size]

            # If all beams have ended, stop early
            if all(tokens[-1] == eos_idx for _, tokens in beam):
                completed.extend(beam)
                break

        # Add any remaining beam entries that didn't finish
        completed.extend(beam)

    # Pick the sequence with the highest score
    best_score, best_tokens = max(completed, key=lambda x: x[0])

    # Decode, skip special tokens
    words = [
        idx2word.get(idx, '<UNK>')
        for idx in best_tokens
        if idx not in {sos_idx, eos_idx, word2idx['<PAD>']}
    ]
    return ' '.join(words)


print('Greedy decode function ready ✓')
print('Beam search decode function ready ✓')

# ── Quick smoke test using the UNTRAINED model (output will be random words) ──
# Build idx2word from word2idx
idx2word = {v: k for k, v in word2idx.items()}

test_img   = [train_dataset[0]['image']]
test_feats = extract_patch_tokens(test_img, vit_model, image_processor, device)  # [1, 196, 768]

greedy_cap = greedy_decode(model_patch, test_feats, word2idx, idx2word)
beam_cap   = beam_search_decode(model_patch, test_feats, word2idx, idx2word, beam_size=3)

print(f'\n[Untrained model — random output expected]')
print(f'Greedy: {greedy_cap}')
print(f'Beam:   {beam_cap}')
print(f'GT:     {train_dataset[0]["caption_text"]}')


Greedy decode function ready ✓
Beam search decode function ready ✓

[Untrained model — random output expected]
Greedy: 
Beam:   collage collage vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary vocabulary
GT:     A black dog is running after a white dog in the snow .



**Next steps → Part 4:** Training loop with Cross-Entropy loss, teacher forcing, and BLEU evaluation.
